GEN-AI ASSIGNMENT 2 MoE
---
HEMANG RAJ SRN: PES2UG23CS219 SECTION: D

1: Environment & Setup

In [6]:
# !pip install groq python-dotenv 

import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables
load_dotenv()

# Initialize Groq client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Standardizing on Llama 3.3 for both routing and generation
MODEL = "llama-3.3-70b-versatile"

2: Define the Experts

In [7]:
MODEL_CONFIG = {
    "technical": {
        "system": "You are a senior technical support engineer. Be rigorous, code-focused, and precise. If applicable, provide well-formatted code snippets to solve the problem.",
        "temperature": 0.7
    },
    "billing": {
        "system": "You are an empathetic billing support specialist. Focus on company policies, financial clarity, and reassuring the customer. Be polite and professional.",
        "temperature": 0.7
    },
    "general": {
        "system": "You are a helpful and friendly general support assistant. Provide brief and courteous answers.",
        "temperature": 0.7
    }
}

3: The Router

In [8]:
def route_prompt(user_input: str) -> str:
    routing_prompt = f"""Classify the following user input into exactly one of these categories: [technical, billing, general].
    Return ONLY the category name as a single lowercase word. Do not add punctuation, reasoning, or explanation.
    
    User Input: '{user_input}'"""
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": routing_prompt}],
        temperature=0, 
        max_tokens=10  
    )
    
    # Clean the output just in case the LLM includes spaces or weird casing
    category = response.choices[0].message.content.strip().lower()
    
    # Fallback safety: If the LLM hallucinates a category, default to general
    if category not in MODEL_CONFIG:
        return "general"
        
    return category

4: The Orchestrator

In [9]:
def process_request(user_input: str) -> str:
    # 1. Decide the category
    category = route_prompt(user_input)
    print(f"[System Log: Router classified intent as -> {category.upper()}]")
    
    # 2. Fetch the appropriate expert configuration
    expert_config = MODEL_CONFIG[category]
    
    # 3. Generate the response using the expert's persona
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": expert_config["system"]},
            {"role": "user", "content": user_input}
        ],
        temperature=expert_config["temperature"]
    )
    
    return response.choices[0].message.content

5: Test Execution

In [10]:
test_queries = [
    "My python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "Hey there, what are your business hours?"
]

for query in test_queries:
    print(f"User: {query}")
    print(f"Agent:\n{process_request(query)}")
    print("-" * 60)

User: My python script is throwing an IndexError on line 5.
[System Log: Router classified intent as -> TECHNICAL]
Agent:
To help you resolve the `IndexError` issue, I'll need more information about your Python script. However, I can provide a general outline of how to debug and potentially fix this issue.

### Debugging Steps

1. **Understand the Error**: An `IndexError` occurs when you try to access an element in a sequence (like a list, tuple, or string) using an index that does not exist.
2. **Review the Code**: Look at line 5 of your script and identify the line of code causing the error. Check if you are trying to access an element in a sequence.
3. **Check Index Values**: Verify that the index you are using is within the valid range for the sequence. Remember that indexing in Python starts at 0, so the last valid index is always one less than the length of the sequence.

### Example of How to Fix an IndexError

Let's say you have a list `my_list` and you're trying to access an e

6: Creating the Mock Tool

In [11]:
def get_current_bitcoin_price() -> str:
    """Mock function to fetch real-time data."""
    # Simulating an API call
    return "$92,450.00 USD"

7: Update the Router & Config

In [12]:
# Add the tool to our existing config
MODEL_CONFIG["data"] = {
    "type": "tool",
    "function": get_current_bitcoin_price
}

def route_prompt_v2(user_input: str) -> str:
    # We added 'data' to the list and gave a tiny instruction on when to use it
    routing_prompt = f"""Classify the following user input into exactly one of these categories: [technical, billing, general, data].
    Use 'data' if the user is asking for real-time statistics, prices, or live metrics.
    Return ONLY the category name as a single lowercase word. Do not add punctuation.
    
    User Input: '{user_input}'"""
    
    response = client.chat.completions.create(
        model=MODEL, # Still using Llama 3.3
        messages=[{"role": "user", "content": routing_prompt}],
        temperature=0, 
        max_tokens=10  
    )
    
    category = response.choices[0].message.content.strip().lower()
    return category if category in MODEL_CONFIG else "general"

8: The "Smart" Orchestrator

In [14]:
def process_request_v2(user_input: str) -> str:
    category = route_prompt_v2(user_input)
    print(f"[System Log: Router classified intent as -> {category.upper()}]")
    
    # 1. Handle the Tool Route
    if category == "data":
        # Execute the python function directly
        tool_result = MODEL_CONFIG["data"]["function"]()
        return f"System Data Retrieved: {tool_result}"
    
    # 2. Handle the standard Prompt Routes
    expert_config = MODEL_CONFIG[category]
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": expert_config["system"]},
            {"role": "user", "content": user_input}
        ],
        temperature=expert_config["temperature"]
    )
    
    return response.choices[0].message.content

9: Test the Tool Route

In [15]:
bonus_queries = [
    "What is the current price of Bitcoin?",
    "How do I reset my password?"
]

for query in bonus_queries:
    print(f"User: {query}")
    print(f"Agent:\n{process_request_v2(query)}")
    print("-" * 60)

User: What is the current price of Bitcoin?
[System Log: Router classified intent as -> DATA]
Agent:
System Data Retrieved: $92,450.00 USD
------------------------------------------------------------
User: How do I reset my password?
[System Log: Router classified intent as -> GENERAL]
Agent:
To reset your password, please go to the login page, click on "Forgot Password," and follow the instructions to enter your email address or username. You'll then receive an email with a link to create a new password. If you need more help, feel free to ask.
------------------------------------------------------------


## 🔬 Technical Observations & Architecture Summary



* **Deterministic Routing:** The Router LLM uses `temperature=0` and `max_tokens=10` for strict, zero-shot classification. This forces predictability in the control flow and minimizes token waste.
* **Token & Latency Efficiency:** By dynamically loading domain-specific System Prompts *after* intent classification, the system avoids stuffing the context window with a massive, generalized "mega-prompt" for every single query. 
* **Stochastic Generation:** The Expert configurations use `temperature=0.7` to allow for flexible, human-like, and contextually appropriate generation once the domain is locked.
* **Deterministic Tool Interception (Bonus):** The `data` route demonstrates an "early-exit" strategy. By intercepting the route and executing a native Python function, we bypass the second LLM inference cycle entirely. This guarantees zero hallucination for objective data requests and saves API compute costs.